# Fragsweep — Main fragmentation sweep, all five models

This notebook runs the full fragmentation sweep used throughout the manuscript: linear SVM, random
forest, VGG-16 (1D), Inception-v1 (1D) and ResNet-18 (1D), each trained and evaluated at seven
fragment lengths (300, 400, 500, 750, 1000, 1500 bp, and full length), on the CD-HIT-97%-identity
grouped split that this study treats as primary everywhere.

**This notebook is resume-safe.** Before training anything, it loads any existing
`fragsweep_all_raw.csv` already in `/kaggle/working` (or that you re-upload) and skips any
(length, seed, identity, model) combination already present. This matters because, as the
runtime estimate below shows, the full grid will almost certainly need more than one Kaggle
session.

**GPU needed.** Yes. In this notebook, 3 seeds x 5 encodings x 1 architecture took roughly 5-8
hours per architecture at full length alone. Across the full 7-length grid, expect each (length,
seed, architecture) fit to take very roughly 15-30 minutes. **`FAST_MODE = True` (default) runs
2 seeds x 4 lengths x all 3 CNNs = 24 fits, roughly 6-12 hours -- still likely more than one
session.** Run it in chunks: start it, let it checkpoint incrementally, stop when your
session/quota runs low, and re-run later -- already-completed cells are skipped automatically.
Only flip to the full 3-seed x 7-length grid once you've confirmed the reduced grid's numbers
look sane and you have GPU quota to spare.

**Dataset.** All notebooks below read the *same* dataset you already used for the main experiments and the 5-model fragment sweep:

`/kaggle/input/datasets/yashrajrajarishi/subtype-classification/prepared_sequences_clean.csv`

with columns `sequence`, `subtype`, `cluster` (the MinHash cluster id). Attach that Kaggle Dataset to the notebook the same way you did for the fragment-test notebook (Add Data -> Your Datasets -> subtype-classification). **No new download is needed** for this file. Where a test needs something extra (a reference genome, an external tool), that is called out explicitly at the top of the notebook, with the exact cell that fetches or installs it -- it downloads automatically inside the notebook, you do not need to upload anything yourself.


**Kaggle session budget.** Kaggle notebook sessions cap out at **12 hours** and (for GPU) a **~30 GPU-hours/week** quota. Each notebook below states whether it needs a GPU and gives a rough runtime estimate at the default settings, plus a `FAST_MODE` toggle to shrink seeds/epochs/lengths if you are short on quota. If a run is approaching the 12 h wall, results are saved to `/kaggle/working` incrementally (per seed / per length) so a partial run is still usable -- just re-run with the remaining settings and concatenate the CSVs.


### 1. Imports and install CD-HIT

In [ ]:

import warnings; warnings.filterwarnings("ignore")
import os, re, time, subprocess, shutil, random
import numpy as np, pandas as pd
import tensorflow as tf

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, balanced_accuracy_score, accuracy_score, matthews_corrcoef

from tensorflow.keras.layers import (Input, Conv1D, BatchNormalization, ReLU, Add,
    GlobalAveragePooling1D, Dropout, Dense, MaxPooling1D,
    SpatialDropout1D, Layer, Concatenate, MaxPool1D)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical

OUT = "/kaggle/working"
os.makedirs(OUT, exist_ok=True)

if shutil.which("cd-hit-est") is None:
    print("cd-hit-est not found, installing...")
    subprocess.run("conda install -y -c bioconda cd-hit 2>/dev/null || true", shell=True)
    if shutil.which("cd-hit-est") is None:
        subprocess.run(
            "wget -q https://github.com/weizhongli/cdhit/releases/download/V4.8.1/cd-hit-v4.8.1-2019-0228.tar.gz "
            "&& tar xzf cd-hit-v4.8.1-2019-0228.tar.gz && cd cd-hit-v4.8.1-2019-0228 && make -j4 "
            "&& cp cd-hit-est /usr/local/bin/ && cp cd-hit /usr/local/bin/", shell=True)
print("cd-hit-est available:", shutil.which("cd-hit-est") is not None)


### 2. Config -- READ THIS before you Run All

In [ ]:

FAST_MODE       = False            # False: full 3 seeds, 7 lengths (0,1,2; 300/400/500/750/1000/1500/full) -- the full-replication rerun
RUN_CD95_TOO    = False
RUN_DEEP_MODELS = True              # False: classical-only (SVM/RF/Kameris-eq/CASTOR-eq), no GPU, <1h

SEEDS   = [0, 1] if FAST_MODE else [0, 1, 2]
LENGTHS = [300, 500, 1000, None] if FAST_MODE else [300, 400, 500, 750, 1000, 1500, None]
DEEP_MODELS = ["resnet18", "vgg16", "inception"]   # kept all three -- none is safe to drop here
MAX_TFIDF_FEATS = 512
IDENTITIES = [0.97] + ([0.95] if RUN_CD95_TOO else [])

EPOCHS = 120     # matches the original per-architecture notebooks
BATCH  = 48


### 3. Load data and write FASTA for CD-HIT

In [ ]:

DATA_PATH = "/kaggle/input/datasets/yashrajrajarishi/subtype-classification/prepared_sequences_clean.csv"
df = pd.read_csv(DATA_PATH).reset_index(drop=True)

le = LabelEncoder(); y_all = le.fit_transform(df["subtype"].values)
CLASSES = list(le.classes_); n_cls = len(CLASSES)
X_all = df["sequence"].values
n = len(df)
print(f"{n} sequences, {n_cls} classes: {CLASSES}")

with open(f"{OUT}/pol_fragsweep.fasta", "w") as f:
    for i, s in enumerate(X_all):
        f.write(f">{i}\n{s}\n")


### 4. Run CD-HIT and parse cluster assignments

In [ ]:

def run_cdhit(identity, word_size):
    out = f"{OUT}/pol_fragsweep_cdhit_{int(identity*100)}"
    cmd = f"cd-hit-est -i {OUT}/pol_fragsweep.fasta -o {out} -c {identity} -n {word_size} -M 16000 -T 4 -d 0"
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return out, r.returncode

def parse_cdhit_clusters(clstr_path, n_seqs):
    labels = np.full(n_seqs, -1, dtype=int)
    cid = -1
    with open(clstr_path) as f:
        for line in f:
            if line.startswith(">Cluster"):
                cid += 1
            else:
                idx = int(line.split(">")[1].split("...")[0])
                labels[idx] = cid
    assert (labels >= 0).all()
    return labels

WORD_SIZE = {0.95: 10, 0.97: 10}
cdhit_groups = {}
for identity in IDENTITIES:
    out, rc = run_cdhit(identity, WORD_SIZE[identity])
    assert rc == 0 and os.path.exists(out + ".clstr"), f"CD-HIT failed at identity={identity}"
    labs = parse_cdhit_clusters(out + ".clstr", n)
    cdhit_groups[identity] = labs
    print(f"CD-HIT {identity*100:.0f}% identity: {len(set(labs))} clusters, "
          f"largest = {pd.Series(labs).value_counts().iloc[0]} sequences")


### 5. Fragmentation + split helpers

In [ ]:

def fragment(seq, L, rng):
    if L is None or len(seq) <= L:
        return seq
    start = rng.integers(0, len(seq) - L + 1)
    return seq[start:start+L]

def split_grouped(X, y, grp, seed):
    s1 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    trv, te = next(s1.split(X, y, grp))
    s2 = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=seed)
    tr_rel, va_rel = next(s2.split(X[trv], y[trv], grp[trv]))
    return trv[tr_rel], trv[va_rel], te


### 6. Resume support -- load any existing results and skip what's already done

In [ ]:

RAW_PATH = f"{OUT}/fragsweep_all_raw.csv"
if os.path.exists(RAW_PATH):
    existing = pd.read_csv(RAW_PATH)
    done_keys = set(zip(existing["length"].astype(str), existing["seed"], existing["identity"], existing["model"]))
    print(f"Resuming: found {len(existing)} existing rows, {len(done_keys)} unique (length,seed,identity,model) combos already done.")
else:
    existing = pd.DataFrame()
    done_keys = set()
    print("No existing results found -- starting fresh.")


### 7. Classical models at every length: SVM, RF, Kameris-equivalent, CASTOR-equivalent -- CPU only

In [ ]:

def fit_tfidf(seq_tr, seq_va, seq_te):
    vec = TfidfVectorizer(analyzer="char", ngram_range=(6,6), max_features=MAX_TFIDF_FEATS, lowercase=False)
    Xtr = vec.fit_transform(seq_tr); Xva = vec.transform(seq_va); Xte = vec.transform(seq_te)
    return Xtr, Xva, Xte

def kameris_features(seqs):
    vec = CountVectorizer(analyzer="char", ngram_range=(6,6), lowercase=False)
    X = vec.fit_transform(seqs)
    X = X.multiply(1.0 / np.maximum(X.sum(axis=1), 1))
    return vec, X.tocsr()

def kameris_transform(vec, seqs):
    X = vec.transform(seqs)
    X = X.multiply(1.0 / np.maximum(X.sum(axis=1), 1))
    return X.tocsr()

IUPAC = {"A":"A","C":"C","G":"G","T":"T","R":"[AG]","Y":"[CT]","S":"[GC]","W":"[AT]",
         "K":"[GT]","M":"[AC]","B":"[CGT]","D":"[AGT]","H":"[ACT]","V":"[ACG]","N":"[ACGT]"}
ENZYMES = {
 "EcoRI":"GAATTC","BamHI":"GGATCC","HindIII":"AAGCTT","PstI":"CTGCAG","SalI":"GTCGAC",
 "XhoI":"CTCGAG","NotI":"GCGGCCGC","SmaI":"CCCGGG","KpnI":"GGTACC","SacI":"GAGCTC",
 "SpeI":"ACTAGT","NheI":"GCTAGC","XbaI":"TCTAGA","ClaI":"ATCGAT","EcoRV":"GATATC",
 "NcoI":"CCATGG","NdeI":"CATATG","AflII":"CTTAAG","ApaI":"GGGCCC","BglII":"AGATCT",
 "DraI":"TTTAAA","HaeIII":"GGCC","HhaI":"GCGC","MspI":"CCGG","RsaI":"GTAC",
 "TaqI":"TCGA","AluI":"AGCT","MboI":"GATC","Sau3AI":"GATC","HpaII":"CCGG",
 "PvuII":"CAGCTG","ScaI":"AGTACT","StuI":"AGGCCT","AvaI":"CYCGRG","BstEII":"GGTNACC",
 "DpnI":"GATC","FokI":"GGATG","MseI":"TTAA","NlaIII":"CATG","Sau96I":"GGNCC",
}
ENZYME_REGEX = {name: re.compile("".join(IUPAC[c] for c in site)) for name, site in ENZYMES.items()}

def rflp_features(seq):
    feats = []; n_ = len(seq)
    for name, rx in ENZYME_REGEX.items():
        cuts = [m.start() for m in rx.finditer(seq)]
        n_cuts = len(cuts)
        frag_lens = [n_] if n_cuts == 0 else np.diff([0]+cuts+[n_]).tolist()
        rms = float(np.sqrt(np.mean(np.square(frag_lens))))
        feats.extend([n_cuts, rms])
    return feats

def build_rflp_matrix(seqs):
    return np.array([rflp_features(s) for s in seqs], dtype=np.float32)

rows = [] if existing.empty else existing.to_dict("records")
splits_cache = {}

for L in LENGTHS:
    tag = L if L else "full"
    for seed in SEEDS:
        rng = np.random.default_rng(seed)
        seqs = np.array([fragment(s, L, rng) for s in X_all])

        for identity, groups in cdhit_groups.items():
            tr, va, te = split_grouped(seqs, y_all, groups, seed)
            splits_cache[(tag, seed, identity)] = (tr, va, te)

            classical_models = ["svm", "rf", "kameris_equivalent", "castor_equivalent"]
            if all((str(tag), seed, identity, m) in done_keys for m in classical_models):
                continue

            Xtr, Xva, Xte = fit_tfidf(seqs[tr], seqs[va], seqs[te])
            for m in ["svm", "rf"]:
                if (str(tag), seed, identity, m) in done_keys:
                    continue
                clf = LinearSVC(class_weight="balanced", random_state=seed, max_iter=20000) if m == "svm" \
                      else RandomForestClassifier(n_estimators=400, class_weight="balanced", random_state=seed, n_jobs=-1)
                clf.fit(Xtr, y_all[tr]); pred = clf.predict(Xte)
                rows.append(dict(length=tag, seed=seed, identity=identity, model=m,
                                  f1=f1_score(y_all[te], pred, average="macro"),
                                  bal_acc=balanced_accuracy_score(y_all[te], pred)))

            if (str(tag), seed, identity, "kameris_equivalent") not in done_keys:
                kvec, Ktr = kameris_features(seqs[tr]); Kte = kameris_transform(kvec, seqs[te])
                clf = LinearSVC(class_weight="balanced", random_state=seed, max_iter=20000)
                clf.fit(Ktr, y_all[tr]); pred = clf.predict(Kte)
                rows.append(dict(length=tag, seed=seed, identity=identity, model="kameris_equivalent",
                                  f1=f1_score(y_all[te], pred, average="macro"),
                                  bal_acc=balanced_accuracy_score(y_all[te], pred)))

            if (str(tag), seed, identity, "castor_equivalent") not in done_keys:
                Rtr = build_rflp_matrix(seqs[tr]); Rte = build_rflp_matrix(seqs[te])
                clf = RandomForestClassifier(n_estimators=400, class_weight="balanced", random_state=seed, n_jobs=-1)
                clf.fit(Rtr, y_all[tr]); pred = clf.predict(Rte)
                rows.append(dict(length=tag, seed=seed, identity=identity, model="castor_equivalent",
                                  f1=f1_score(y_all[te], pred, average="macro"),
                                  bal_acc=balanced_accuracy_score(y_all[te], pred)))

        pd.DataFrame(rows).to_csv(RAW_PATH, index=False)
        print(f"[classical] length={tag} seed={seed} done")


### 8. CNN architectures (ResNet-18, VGG-16, Inception-v1)

In [ ]:

class StochasticDepth(Layer):
    def __init__(self, p=0.85, **kw): super().__init__(**kw); self.p = p
    def call(self, x, training=None):
        if not training: return x
        r = tf.random.uniform([tf.shape(x)[0], 1, 1])
        return x * tf.floor(self.p + r) / self.p
    def get_config(self): return {**super().get_config(), "p": self.p}

def res_block(x, f, stride=1, sp=0.85, dr=0.1):
    sc = x
    x = Conv1D(f, 3, strides=stride, padding="same", use_bias=False)(x)
    x = ReLU()(BatchNormalization()(x)); x = SpatialDropout1D(dr)(x)
    x = BatchNormalization()(Conv1D(f, 3, padding="same", use_bias=False)(x))
    if stride != 1 or sc.shape[-1] != f:
        sc = BatchNormalization()(Conv1D(f, 1, strides=stride, padding="same", use_bias=False)(sc))
    return ReLU()(Add()([StochasticDepth(sp)(x), sc]))

def build_resnet18(input_len, n_classes):
    inp = Input((input_len, 1)); x = inp
    x = ReLU()(BatchNormalization()(Conv1D(32, 7, strides=2, padding="same", use_bias=False)(x)))
    x = MaxPooling1D(3, strides=2, padding="same")(x)
    for f, sp in [(32, .95), (32, .92)]:  x = res_block(x, f, 1, sp)
    for f, sp in [(64, .88), (64, .85)]:  x = res_block(x, f, 2 if f != x.shape[-1] else 1, sp)
    for f, sp in [(128, .80), (128, .78)]: x = res_block(x, f, 2 if f != x.shape[-1] else 1, sp)
    for f, sp in [(256, .75), (256, .72)]: x = res_block(x, f, 2 if f != x.shape[-1] else 1, sp)
    x = Dropout(0.4)(GlobalAveragePooling1D()(x))
    out = Dense(n_classes, activation="softmax")(x)
    m = Model(inp, out)
    m.compile(optimizer=Adam(3e-4), loss=CategoricalCrossentropy(label_smoothing=0.1), metrics=["accuracy"])
    return m

def build_vgg16(input_len, n_classes):
    def conv_block(x, f, n_convs):
        for _ in range(n_convs):
            x = ReLU()(BatchNormalization()(Conv1D(f, 3, padding="same", use_bias=False)(x)))
        return MaxPooling1D(2, padding="same")(x)
    inp = Input((input_len, 1))
    x = conv_block(inp, 32, 2)
    x = conv_block(x,   64, 2)
    x = conv_block(x,  128, 3)
    x = conv_block(x,  256, 3)
    x = conv_block(x,  256, 3)
    x = Dropout(0.4)(GlobalAveragePooling1D()(x))
    x = Dropout(0.4)(ReLU()(Dense(256)(x)))
    out = Dense(n_classes, activation="softmax")(x)
    m = Model(inp, out)
    m.compile(optimizer=Adam(3e-4), loss=CategoricalCrossentropy(label_smoothing=0.1), metrics=["accuracy"])
    return m

def inception_module(x, f1, f3r, f3, f5r, f5, fp):
    b1 = Conv1D(f1, 1, padding="same", activation="relu")(x)
    b3 = Conv1D(f3, 3, padding="same", activation="relu")(Conv1D(f3r, 1, padding="same", activation="relu")(x))
    b5 = Conv1D(f5, 5, padding="same", activation="relu")(Conv1D(f5r, 1, padding="same", activation="relu")(x))
    bp = Conv1D(fp, 1, padding="same", activation="relu")(MaxPooling1D(3, strides=1, padding="same")(x))
    return Concatenate()([b1, b3, b5, bp])

def build_inception(input_len, n_classes):
    inp = Input((input_len, 1))
    x = MaxPooling1D(3, strides=2, padding="same")(
            ReLU()(BatchNormalization()(Conv1D(32, 7, strides=2, padding="same")(inp))))
    x = inception_module(x, 16, 24, 32, 4,  8,  8)
    x = inception_module(x, 32, 32, 48, 8, 24, 16)
    x = MaxPooling1D(3, strides=2, padding="same")(x)
    x = inception_module(x, 48, 24, 52, 8, 24, 16)
    x = Dropout(0.4)(GlobalAveragePooling1D()(x))
    out = Dense(n_classes, activation="softmax")(x)
    m = Model(inp, out)
    m.compile(optimizer=Adam(3e-4), loss=CategoricalCrossentropy(label_smoothing=0.1), metrics=["accuracy"])
    return m

BUILDERS = {"resnet18": build_resnet18, "vgg16": build_vgg16, "inception": build_inception}

def run_deep_original(model_name, Xtr, ytr, Xva, yva, Xte, yte, seed, n_classes):
    """Exact training recipe from the original notebooks: per-feature StandardScaler
    (fit on train only), one-hot labels, CategoricalCrossentropy(label_smoothing=0.1),
    Adam(3e-4), up to 120 epochs, EarlyStopping(patience=15) + ReduceLROnPlateau."""
    tf.random.set_seed(seed)
    sc = StandardScaler()
    Xtr_d = sc.fit_transform(np.asarray(Xtr.todense() if hasattr(Xtr, "todense") else Xtr))[..., None]
    Xva_d = sc.transform(np.asarray(Xva.todense() if hasattr(Xva, "todense") else Xva))[..., None]
    Xte_d = sc.transform(np.asarray(Xte.todense() if hasattr(Xte, "todense") else Xte))[..., None]
    cw = dict(enumerate(compute_class_weight("balanced", classes=np.unique(ytr), y=ytr)))
    model = BUILDERS[model_name](Xtr_d.shape[1], n_classes)
    model.fit(Xtr_d, to_categorical(ytr, n_classes),
              validation_data=(Xva_d, to_categorical(yva, n_classes)),
              epochs=EPOCHS, batch_size=BATCH, class_weight=cw, shuffle=True, verbose=0,
              callbacks=[EarlyStopping("val_loss", patience=15, restore_best_weights=True),
                         ReduceLROnPlateau("val_loss", factor=0.5, patience=6, min_lr=1e-6)])
    prob = model.predict(Xte_d, verbose=0)
    pred = prob.argmax(1)
    f1 = f1_score(yte, pred, average="macro"); bal = balanced_accuracy_score(yte, pred)
    tf.keras.backend.clear_session()
    return f1, bal, pred


### 9. CNNs at every length -- original architectures, resume-safe

In [ ]:

fragsweep_classical = pd.DataFrame(rows)

if RUN_DEEP_MODELS:
    deep_rows = [r for r in rows if r.get("model") in DEEP_MODELS]
    for (tag, seed, identity), (tr, va, te) in splits_cache.items():
        remaining = [m for m in DEEP_MODELS if (str(tag), seed, identity, m) not in done_keys]
        if not remaining:
            continue
        L = None if tag == "full" else tag
        rng = np.random.default_rng(seed)
        seqs = np.array([fragment(s, L, rng) for s in X_all])
        Xtr, Xva, Xte = fit_tfidf(seqs[tr], seqs[va], seqs[te])
        for m in remaining:
            f1, bal, pred = run_deep_original(m, Xtr, y_all[tr], Xva, y_all[va], Xte, y_all[te], seed, n_cls)
            deep_rows.append(dict(length=tag, seed=seed, identity=identity, model=m, f1=f1, bal_acc=bal))
            pd.DataFrame(rows + deep_rows).to_csv(f"{OUT}/fragsweep_all_raw.csv", index=False)   # incremental save
            print(f"[deep] length={tag} seed={seed} identity={identity} model={m} f1={f1:.4f} done")

    fragsweep_all = pd.DataFrame(rows + deep_rows)
else:
    print("RUN_DEEP_MODELS is False -- skipped. fragsweep_all is classical-only.")
    fragsweep_all = fragsweep_classical

fragsweep_all.to_csv(f"{OUT}/fragsweep_all_raw.csv", index=False)


### 10. Summary table

In [ ]:

summary = (fragsweep_all[fragsweep_all["identity"]==0.97]
           .groupby(["length","model"])["f1"].agg(["mean","std"])
           .reset_index().pivot(index="length", columns="model", values=["mean","std"]))
summary.to_csv(f"{OUT}/fragsweep_summary_table_cd97.csv")
print("CD-HIT 97% identity (primary split), mean +/- std macro-F1:")
print(summary.round(4))


In [ ]:
!cd /kaggle/working && zip -r fragsweep_fragmentation_rerun_outputs.zip fragsweep_all_raw.csv fragsweep_summary_table_cd97.csv 2>/dev/null; ls -la /kaggle/working/*.zip